# Урок 16. Файловые системы и хранение данных

10 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [← Урок 15](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-15.ipynb) · [Урок 17 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-17.ipynb)

---

Организация файловой системы. Пути, маски, права доступа. Объём и фрагментация. Резервное копирование. Форматы хранения данных.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 10А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="10-16", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Что такое файловая система

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g10/disk-vnutri.jpg" width="380" alt="Внутри жёсткого диска: пластины и головка">

*Внутри жёсткого диска: пластины и головка*

<sub>Eric Gaba (Sting) · CC BY-SA 3.0 · Wikimedia Commons</sub>

Диск не знает ни про файлы, ни про папки. Он умеет одно: прочитать
или записать блок байтов по номеру. Всё остальное — договорённость,
которую поддерживает **файловая система**: как из пронумерованных
блоков собрать файл, где хранить имена и где записано, какие блоки
свободны.

Наиболее известные файловые системы: NTFS (Windows), ext4 (Linux),
APFS (macOS), exFAT (флешки, понимают все системы).

### Из чего состоит файловая система

| Часть | Что хранит |
|---|---|
| таблица файлов | имя, размер, даты, права, список блоков |
| каталоги | какие файлы лежат внутри какой папки |
| карта свободного места | какие блоки заняты, какие свободны |
| сами данные | содержимое файлов, разложенное по блокам |

### Кластер и потерянное место

Диск размечают на **кластеры** — минимальные порции, которые может
занять файл. Обычно 4 килобайта. Файл в 1 байт всё равно занимает
целый кластер, а файл в 5 килобайт — два, и второй заполнен
на четверть.

Потерянное место называют внутренней фрагментацией. Для тысячи
крошечных файлов оно заметно: 1000 файлов по 100 байт займут
4 мегабайта вместо ста килобайт.

### Фрагментация

Когда файл записывают, ему выдают свободные блоки. Если подходящего
сплошного места нет, файл раскладывают по кускам в разных местах
диска — он становится **фрагментированным**.

Для жёсткого диска это беда: головке приходится прыгать, а каждый
прыжок — миллисекунды. Отсюда и появилась дефрагментация. Для SSD
она не нужна и даже вредна: доступ к любой ячейке одинаково быстрый,
а лишние перезаписи сокращают срок службы.

### Пути

**Абсолютный путь** начинается от корня и однозначно определяет файл:

```
Linux и Colab:   /content/данные/отчёт.txt
Windows:         C:\Документы\отчёт.txt
```

**Относительный путь** отсчитывается от текущей папки:

| Запись | Значение |
|---|---|
| `отчёт.txt` | файл в текущей папке |
| `данные/отчёт.txt` | в подпапке `данные` |
| `../отчёт.txt` | на уровень выше |
| `./отчёт.txt` | явно в текущей папке |

В Python пути пишут через прямую косую черту даже на Windows —
так меньше путаницы с обратной чертой, которая в строках означает
служебный символ.

### Маски

Маска — шаблон имени файла. В ЕГЭ и в командах системы работают
два символа:

| Символ | Значение |
|:-:|---|
| `?` | ровно один любой символ |
| `*` | любая последовательность символов, в том числе пустая |

Примеры:

| Маска | Подходит | Не подходит |
|---|---|---|
| `*.txt` | `а.txt`, `отчёт.txt` | `а.txtx`, `txt` |
| `?a*.t?t` | `ba.tit`, `xattt.txt` | `a.txt` (первый символ не любой + a) |
| `дом*.??` | `дом.py`, `домик.md` | `дом.txt` |

Разбирать такие задания надо по частям: сначала расширение, потом
начало имени, потом длина.

### Права доступа

В Unix-системах у каждого файла три группы прав — для владельца,
группы и всех остальных, и три вида доступа: чтение (r), запись (w),
выполнение (x). Запись `rwxr-xr--` означает: владельцу всё, группе
читать и выполнять, остальным только читать.

### Резервное копирование

Правило «3-2-1»: три копии данных, на двух разных носителях, одна —
в другом месте. Скучно ровно до первого отказа диска.

Отдельный вид копии — **версии**: не просто последняя копия, а история
изменений. Именно так работают облачные диски и системы контроля
версий вроде Git, на котором держится наш учебник.

## Смотрим, как это работает

### Пример 1. Сколько места займут файлы

In [ ]:
def занято_кластеров(размер_байт, кластер=4096):
    if размер_байт == 0:
        return 1
    return -(-размер_байт // кластер)          # деление с округлением вверх


файлы = [100, 4096, 4097, 10_000, 1_000_000]

print(f"{'размер, байт':>14} {'кластеров':>10} {'занято, байт':>14} {'потеряно':>10}")
for размер in файлы:
    кластеров = занято_кластеров(размер)
    занято = кластеров * 4096
    print(f"{размер:>14,} {кластеров:>10} {занято:>14,} {занято - размер:>10,}".replace(",", " "))

Приём `-(-a // b)` — деление с округлением вверх без обращения
к модулю `math`. Обычное `//` округляет вниз, два минуса
переворачивают это дважды.

### Пример 2. Тысяча маленьких файлов

In [ ]:
для_файлов = 1000
размер_каждого = 100

реально = для_файлов * размер_каждого
на_диске = для_файлов * занято_кластеров(размер_каждого) * 4096

print(f"Данных:    {реально / 1024:.1f} КБ")
print(f"На диске:  {на_диске / 1024 / 1024:.1f} МБ")
print(f"Потеряно:  {(на_диске - реально) / на_диске * 100:.1f}%")

Поэтому архив из тысячи мелких файлов не только удобнее копировать,
но и занимает в разы меньше места.

### Пример 3. Маски файлов

Модуль `fnmatch` умеет проверять имя по маске теми же правилами,
что и ЕГЭ.

In [ ]:
from fnmatch import fnmatch

имена = ["отчёт.txt", "отчёт.txtx", "документ.txt", "a.txt", "ba.tit",
         "дом.py", "домик.md", "дом.txt"]

for маска in ("*.txt", "?a*.t?t", "дом*.??"):
    подходят = [имя for имя in имена if fnmatch(имя, маска)]
    print(f"{маска:<10} → {подходят}")

Проверьте вывод по таблице из теории. Обратите внимание на `a.txt`
и маску `?a*.t?t`: знак `?` требует **ровно один** символ перед `a`,
а его нет.

### Пример 4. Разбор пути

In [ ]:
import os

путь = "/content/данные/2026/отчёт.txt"

print("Полный путь:  ", путь)
print("Папка:        ", os.path.dirname(путь))
print("Имя файла:    ", os.path.basename(путь))
print("Только имя:   ", os.path.splitext(os.path.basename(путь))[0])
print("Расширение:   ", os.path.splitext(путь)[1])
print("По частям:    ", путь.strip("/").split("/"))

Разбирать путь вручную через `split` можно, но `os.path` учитывает
особенности каждой системы — с ним программа заработает и в Windows,
и в Linux.

### Пример 5. Обход папки

Создадим несколько файлов и посмотрим на них так, как это делает
файловый менеджер.

In [ ]:
import os

os.makedirs("склад/фото", exist_ok=True)
for имя, содержимое in [
    ("склад/список.txt", "хлеб\nмолоко\n"),
    ("склад/заметка.txt", "не забыть\n"),
    ("склад/данные.csv", "a;b\n1;2\n"),
    ("склад/фото/кот.jpg", "не настоящая картинка"),
]:
    with open(имя, "w", encoding="utf-8") as файл:
        файл.write(содержимое)

for папка, подпапки, файлы in os.walk("склад"):
    for имя in файлы:
        полный = os.path.join(папка, имя)
        print(f"{полный:<28} {os.path.getsize(полный):>5} байт")

`os.walk` спускается по всему дереву папок сам — это готовая
рекурсия, которую не надо писать руками.

## Пробуем сами

### Задача 1. Сколько кластеров

Сколько кластеров по 4096 байт займёт файл размером 10 000 байт?

In [ ]:
#@title 🧩 Задача 1. Кластеры { display-mode: "form" }
#@markdown Впишите число
кластеров = 0 #@param {type:"integer"}

si.ответ("1", кластеров, "4e07408562bedb8b",
         hint="10000 / 4096 = 2,44 — и округляем вверх.")

### Задача 2. Функция подсчёта кластеров

Функция получает размер файла в байтах и размер кластера, возвращает
количество занятых кластеров. Пустой файл занимает один кластер.

In [ ]:
def кластеров_нужно(размер, кластер):
    return ...

In [ ]:
si.check("2", кластеров_нужно, [
    ((10000, 4096), 3),
    ((4096, 4096), 1),
    ((4097, 4096), 2),
    ((0, 4096), 1),
    ((1, 512), 1),
])

### Задача 3. Потерянные байты

Функция возвращает, сколько байт пропадёт впустую при хранении файла
заданного размера.

In [ ]:
def потеряно(размер, кластер):
    return ...

In [ ]:
si.check("3", потеряно, [
    ((100, 4096), 3996),
    ((4096, 4096), 0),
    ((5000, 4096), 3192),
    ((0, 512), 512),
])

### Задача 4. Маска

Какое имя файла **не** подходит под маску `?ab*.c?`?

In [ ]:
#@title 🧩 Задача 4. Маска { display-mode: "form" }
#@markdown Выберите ответ
не_подходит = "выбери ответ" #@param ["выбери ответ", "1abc.cd", "xab.cc", "ab.cd", "zabcd.cx"]

si.ответ("4", не_подходит, "1fac6b78ae2dc213",
         hint="Знак ? требует ровно один символ, ни больше ни меньше.")

### Задача 5. Проверка расширения

Функция получает имя файла и расширение (без точки), возвращает
`True`, если имя заканчивается этим расширением. Регистр не важен.

In [ ]:
def это_расширение(имя, расширение):
    return ...

In [ ]:
si.check("5", это_расширение, [
    (("отчёт.txt", "txt"), True),
    (("отчёт.TXT", "txt"), True),
    (("отчёт.txtx", "txt"), False),
    (("txt", "txt"), False),
])

### Задача 6. Имя файла из пути

Функция получает абсолютный путь с разделителем `/` и возвращает
имя файла без папок.

In [ ]:
def имя_файла(путь):
    return ...

In [ ]:
si.check("6", имя_файла, [
    ("/content/данные/отчёт.txt", "отчёт.txt"),
    ("/файл.py", "файл.py"),
    ("просто.txt", "просто.txt"),
])

### Задача 7. Нужна ли дефрагментация

Для какого накопителя дефрагментация полезна?

In [ ]:
#@title 🧩 Задача 7. Дефрагментация { display-mode: "form" }
#@markdown Выберите ответ
кому_нужна = "выбери ответ" #@param ["выбери ответ", "жёсткому диску", "SSD", "обоим одинаково"]

si.ответ("7", кому_нужна, "7f5262438a344738",
         hint="Там, где есть механическая головка.")

## Домашнее задание

### Домашнее задание 1. Место под папку

Функция получает список размеров файлов и размер кластера, возвращает,
сколько байт займут эти файлы на диске.

In [ ]:
def место_на_диске(размеры, кластер):
    return ...

In [ ]:
si.check("дз1", место_на_диске, [
    (([100, 100, 100], 4096), 12288),
    (([4096, 8192], 4096), 12288),
    (([], 4096), 0),
    (([5000], 1024), 5120),
])

### Домашнее задание 2. Отбор по маске

Функция получает список имён и расширение, возвращает список имён
с этим расширением, порядок сохраняется.

In [ ]:
def отобрать(имена, расширение):
    return ...

In [ ]:
si.check("дз2", отобрать, [
    ((["а.txt", "б.py", "в.TXT"], "txt"), ["а.txt", "в.TXT"]),
    ((["а.py"], "txt"), []),
    (([], "txt"), []),
])

### Домашнее задание 3. Ревизия своего диска

Найдите на своём компьютере пять самых больших файлов и пять самых
старых. Запишите, какие из них можно удалить, а какие стоит
скопировать в облако. Заодно проверьте по правилу «3-2-1»: есть ли
у вас хоть одна резервная копия важных данных? На уроке обсудим,
что именно стоит считать важным.

---

### Любопытно

Удаление файла обычно не стирает данные: система лишь помечает
кластеры свободными, а содержимое остаётся, пока его не перезапишут.
Поэтому удалённые файлы часто удаётся восстановить — и поэтому же
перед продажей компьютера диск надёжно затирают. У SSD всё сложнее:
из-за внутреннего перераспределения ячеек часть данных может
оставаться недоступной даже для перезаписи, и единственный
по-настоящему надёжный способ — шифровать диск с самого начала.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 15](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-15.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 17 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-10/urok-17.ipynb)